# W03 — Data Contract: Content Performance (Decline Risk) Lane

**Lane:** content-day performance from the FlyRank warehouse.
**Table(s):** `fact_content_daily_performance` (main), `dim_content`, `dim_clients` (context / availability).
**Dev month:** `month=2026-03` (mid-panel). **Sealed test month:** `2026-06` (only touch via `_sample` for query mechanics, never for label logic).

> If your assigned lane is different (e.g. the query-level lane on `fact_content_query_90d`),
> swap the table name in the setup cell and adjust the grain columns in Query A — the rest of
> this notebook's structure (contract → 3 queries → 5 features → trap → limitation) stays the same.


In [1]:
# Setup — DuckDB reading Parquet directly off the Hugging Face Hub.
# HF_TOKEN must be a Colab Secret (key icon in the left sidebar) — never pasted in a cell.
!pip -q install duckdb --upgrade

import duckdb, os
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_url_style='path';")
con.execute(f"CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN '{HF_TOKEN}');")

BASE = "hf://datasets/FlyRank/internship-warehouse"
DEV_MONTH = "2026-03"     # mid-panel — iterate here
TEST_MONTH_SAMPLE = f"{BASE}/fact_content_daily_performance_sample"  # sealed final month, mechanics only

print("DuckDB ready. Dev month:", DEV_MONTH)


DuckDB ready. Dev month: 2026-03


In [2]:
# One-time discovery: confirm the actual partition path pattern before trusting the glob below.
# Adjust MONTH_GLOB if your repo's file layout differs from this guess.
MONTH_GLOB = f"{BASE}/fact_content_daily_performance/month={DEV_MONTH}/*.parquet"

try:
    n = con.execute(f"SELECT COUNT(*) FROM read_parquet('{MONTH_GLOB}')").fetchone()[0]
    print("Rows found at guessed path:", n)
except Exception as e:
    print("Guessed path didn't resolve — list the dataset repo structure instead:")
    print(e)


Rows found at guessed path: 9841378


## 1)–2) The Contract, in plain words

1. **Unit of analysis (one row means):** one *(client, content item, report_date)* daily performance
   observation — one content item's organic search performance on one calendar day, for one client.

2. **Table(s) used:** `fact_content_daily_performance` for the metric rows, `dim_content` for
   content-level context (`content_created_date`, content type), `dim_clients` for client-level
   context (`gsc_data_start`, `ga4_data_start` — needed to interpret missingness correctly, not
   to model on).

3. **Time window:** develop on `month=2026-03` (mid-panel, has real history behind it and real
   future ahead of it). Treat `month=2026-06` (the `_sample` table) as a **sealed test month** —
   used only to check query mechanics, never to build or tune label logic, per the panel warning.

4. **What we'd predict/rank (label or proxy):** for each content item, a **decline-risk proxy** —
   whether its clicks in the *next* period fall meaningfully below its trailing baseline. This is
   a proxy (not a warehouse-provided label column) computed strictly from *future* rows relative
   to the decision date, which is exactly why it must never leak into the feature side.

5. **One thing deliberately excluded:** raw `ga4_data_available = FALSE` rows' GA4 engagement
   columns are excluded from any feature — they are zero-filled placeholders, not real zero
   engagement, and a blind `fillna(0)` would fabricate a false "no engagement" signal (per the
   flyrank-data skill's panel warning). We keep the availability flag itself but drop the values
   it flags as unavailable.


## 3) Three Verification Queries → Five Features → The Trap

Every claim above gets a query. All three run on `month=2026-03` only.


### First — confirm the real schema (do this once, before trusting anything below)

The column names guessed in an earlier draft (`client_id`, `content_id`, `ga4_data_available`)
were WRONG — DuckDB's own error already proved `client_id`→`client_hash_id` and
`content_id`→`content_hash_id`. The queries below have been updated for those two. But
`report_date`, `gsc_clicks`, `gsc_avg_position`, and the availability flag are still
**unverified guesses** — the error's candidate list showed `client_has_ga4` / `client_has_gsc`
instead of `ga4_data_available`, which may be a different (possibly client-level, not
row-level) column. Run this cell first and read the real names off it before running Query A–C.


In [3]:
schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{MONTH_GLOB}')").fetchdf()
import pandas as pd
pd.set_option('display.max_rows', None)
print(schema.to_string())


                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

### Query A — Grain: is one row really *(client, content, report_date)*?

In [4]:
grain_check = con.execute(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM read_parquet('{MONTH_GLOB}')
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").fetchdf()

print("Duplicate-grain rows found (should be 0):", len(grain_check))
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate-grain rows found (should be 0): 0


,client_hash_id,content_hash_id,report_date,c


### Query B — Row count and date span of this slice

In [5]:
span = con.execute(f"""
    SELECT
        COUNT(*)                        AS n_rows,
        COUNT(DISTINCT client_hash_id)   AS n_clients,
        COUNT(DISTINCT content_hash_id)  AS n_content_items,
        MIN(report_date)                AS min_date,
        MAX(report_date)                AS max_date
    FROM read_parquet('{MONTH_GLOB}')
""").fetchdf()

span


,n_rows,n_clients,n_content_items,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


### Query C — Availability: how many rows survive `IS TRUE`?

**Edit the column name here once you've confirmed it from the schema cell above** — this may
need to become `client_has_ga4` (and possibly joined from `dim_clients` rather than read off
the fact table directly), rather than the guessed `ga4_data_available`.


In [6]:
AVAIL_COL = "ga4_data_available"  # <-- confirm/replace using the schema output above

avail = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN {AVAIL_COL} IS TRUE THEN 1 ELSE 0 END) AS available_rows,
        ROUND(100.0 * SUM(CASE WHEN {AVAIL_COL} IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_available
    FROM read_parquet('{MONTH_GLOB}')
""").fetchdf()

avail
# Expect a meaningful chunk to fail IS TRUE — the skill notes ~1/3 of clients have
# little/no usable GA4 history. This is the number that proves it on THIS slice.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows,pct_available
0,9841378,413966.0,4.2


### Five Features (max) — built on the same `2026-03` slice

Each feature includes its one-line "knowable at the decision moment because…" justification.


**Column names below still assume the guessed `gsc_clicks`, `gsc_avg_position`,
`first_seen_date`, `gsc_data_start`.** Only `client_id`→`client_hash_id` and
`content_id`→`content_hash_id` are confirmed fixes. Check the schema output above and swap
in real names for anything that errors — same pattern as Query A–C.


**First, load the two dimension tables as views** — confirmed against the actual repo file
listing: `dim_content` and `dim_clients` are single files at the repo root
(`dim_content.parquet`, `dim_clients.parquet`), **not** folders of partitioned parquet files
like `fact_content_daily_performance` is. An earlier guess treated them as folders
(`dim_content/*.parquet`) and got a 404 — this is the corrected, verified path. This cell also
prints each table's schema so you can confirm `first_seen_date` / `gsc_data_start` are the real
column names, not more guesses.


In [7]:
con.execute(f"CREATE OR REPLACE VIEW dim_content AS SELECT * FROM read_parquet('{BASE}/dim_content.parquet')")
con.execute(f"CREATE OR REPLACE VIEW dim_clients AS SELECT * FROM read_parquet('{BASE}/dim_clients.parquet')")

print("dim_content columns:")
print(con.execute("DESCRIBE SELECT * FROM dim_content").fetchdf().to_string())
print()
print("dim_clients columns:")
print(con.execute("DESCRIBE SELECT * FROM dim_clients").fetchdf().to_string())


dim_content columns:
                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         

In [8]:
features_df = con.execute(f"""
    WITH base AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            f.report_date,
            f.gsc_clicks,
            f.gsc_avg_position,
            f.ga4_data_available
        FROM read_parquet('{MONTH_GLOB}') f
    )
    SELECT
        b.client_hash_id,
        b.content_hash_id,
        b.report_date,

        -- 1. trailing_clicks_7d: knowable because it's a backward-looking rolling sum,
        --    computed only from days strictly before or equal to report_date.
        SUM(b.gsc_clicks) OVER (
            PARTITION BY b.client_hash_id, b.content_hash_id
            ORDER BY b.report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS trailing_clicks_7d,

        -- 2. prev_avg_position: knowable because it's yesterday's value, not today's or
        --    the future's. (avg_position = 0 means "no data", treated as NULL, not rank 0.)
        LAG(NULLIF(b.gsc_avg_position, 0)) OVER (
            PARTITION BY b.client_hash_id, b.content_hash_id ORDER BY b.report_date
        ) AS prev_avg_position,

        -- 3. content_age_days: knowable because it depends only on the content's creation
        --    date and today's date, both fixed facts at decision time, never on outcomes.
        DATE_DIFF('day', d.content_created_date, b.report_date) AS content_age_days,

        -- 4. client_history_depth_days: knowable because it depends only on the client's
        --    registered gsc_data_start and today's date -- a static, pre-known fact.
        DATE_DIFF('day', c.gsc_data_start, b.report_date) AS client_history_depth_days,

        -- 5. has_ga4_data: knowable because availability is itself known at decision time --
        --    it's a flag about the data, not a value derived from future engagement.
        b.ga4_data_available AS has_ga4_data

    FROM base b
    LEFT JOIN dim_content d ON d.content_hash_id = b.content_hash_id
    LEFT JOIN dim_clients c ON c.client_hash_id = b.client_hash_id
""").fetchdf()

features_df.head(10)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,trailing_clicks_7d,prev_avg_position,content_age_days,client_history_depth_days,has_ga4_data
0,client_0797ff3a1fc9a6a5,content_af9b9b440e25c240,2026-03-01,0.0,NaN,145,116,<NA>
1,client_0797ff3a1fc9a6a5,content_af9b9b440e25c240,2026-03-02,0.0,NaN,146,117,<NA>
2,client_0797ff3a1fc9a6a5,content_af9b9b440e25c240,2026-03-03,0.0,NaN,147,118,<NA>
3,client_0797ff3a1fc9a6a5,content_af9b9b440e25c240,2026-03-04,0.0,NaN,148,119,<NA>
4,client_0797ff3a1fc9a6a5,content_af9b9b440e25c240,2026-03-05,0.0,NaN,149,120,<NA>
5,client_0797ff3a1fc9a6a5,content_af9b9b440e25c240,2026-03-06,0.0,NaN,150,121,<NA>
6,client_0797ff3a1fc9a6a5,content_af9b9b440e25c240,2026-03-07,0.0,NaN,151,122,<NA>
7,client_0797ff3a1fc9a6a5,content_af9b9b440e25c240,2026-03-08,0.0,NaN,152,123,<NA>
8,client_0797ff3a1fc9a6a5,content_af9b9b440e25c240,2026-03-09,0.0,NaN,153,124,<NA>
9,client_0797ff3a1fc9a6a5,content_af9b9b440e25c240,2026-03-10,0.0,NaN,154,125,<NA>


**Available when? — one line each:**

1. `trailing_clicks_7d` — knowable at decision time: a rolling sum over the 7 days up to and
   including `report_date`, never touches a future row.
2. `prev_avg_position` — knowable at decision time: it's the prior day's value via `LAG`, not
   the current or future day's.
3. `content_age_days` — knowable at decision time: derived from a fixed past date
   (`content_created_date`) and today's date only.
4. `client_history_depth_days` — knowable at decision time: derived from a fixed, pre-registered
   client fact (`gsc_data_start`) and today's date only.
5. `has_ga4_data` — knowable at decision time: it's a data-availability flag, not a downstream
   engagement outcome; treated as a feature, its underlying zero-filled values are not.


### The Trap — one label-derived column, on purpose

We define the proxy label as: *clicks in the next 30 days fall below 70% of the trailing 30-day
baseline* (`will_decline`). Then we deliberately add ONE column computed from the **same future
window the label uses** — `next30_clicks` itself — as a "feature", watch a trivial classifier's
score jump toward perfect, then remove it and keep the honest number.


In [9]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Build label: next-30d clicks vs trailing-30d baseline, per (client, content) as of a cut date.
labeled = con.execute(f"""
    WITH daily AS (
        SELECT client_hash_id, content_hash_id, report_date, gsc_clicks
        FROM read_parquet('{MONTH_GLOB}')
    ),
    windowed AS (
        SELECT
            client_hash_id, content_hash_id, report_date, gsc_clicks,
            SUM(gsc_clicks) OVER (
                PARTITION BY client_hash_id, content_hash_id ORDER BY report_date
                ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
            ) AS trail30,
            SUM(gsc_clicks) OVER (
                PARTITION BY client_hash_id, content_hash_id ORDER BY report_date
                ROWS BETWEEN 1 FOLLOWING AND 30 FOLLOWING
            ) AS next30
        FROM daily
    )
    SELECT * FROM windowed WHERE trail30 IS NOT NULL AND next30 IS NOT NULL
""").fetchdf()

labeled["will_decline"] = (labeled["next30"] < 0.7 * labeled["trail30"]).astype(int)
labeled = labeled.merge(features_df, on=["client_hash_id", "content_hash_id", "report_date"], how="inner")

honest_feats = ["trailing_clicks_7d", "prev_avg_position", "content_age_days",
                 "client_history_depth_days", "has_ga4_data"]

def quick_auc(df, feats):
    X = df[feats].copy()
    bool_cols = X.select_dtypes(include=["boolean", "bool"]).columns
    X[bool_cols] = X[bool_cols].astype("Float64")
    X = X.fillna(-1).astype(float)
    y = df["will_decline"]
    if y.nunique() < 2:
        return None
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
    clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    return roc_auc_score(yte, clf.predict_proba(Xte)[:, 1])

honest_auc = quick_auc(labeled, honest_feats)
print("HONEST auc (5 real features):", honest_auc)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

HONEST auc (5 real features): 0.7819038128668553


In [10]:
# Now inject the leak: next30_clicks is literally the future window the label was built from.
labeled["next30_clicks_LEAK"] = labeled["next30"]
leaky_feats = honest_feats + ["next30_clicks_LEAK"]

leaky_auc = quick_auc(labeled, leaky_feats)
print("LEAKY auc (adds next30_clicks_LEAK):", leaky_auc)
print("Jump:", None if honest_auc is None else round(leaky_auc - honest_auc, 4))


LEAKY auc (adds next30_clicks_LEAK): 0.8504105517898259
Jump: 0.0685


In [11]:
# Delete the leak. Keep only the honest number.
labeled = labeled.drop(columns=["next30_clicks_LEAK"])
final_auc = quick_auc(labeled, honest_feats)
print("FINAL honest auc kept for this lane:", final_auc)
assert final_auc == honest_auc, "sanity check: honest score should be unchanged after removing the leak"


FINAL honest auc kept for this lane: 0.7819038128668553


## 4) Named Limitation

**Client history depth is wildly uneven.** `dim_clients.gsc_data_start` varies per client, so
`client_history_depth_days` (feature 4) systematically favors long-tenured clients in the
`2026-03` slice — newer clients look artificially "young" not because their content is new but
because we haven't watched them as long. Any model trained here will underweight or misjudge
recently-onboarded clients until per-client windows (not one global calendar window) are used.
